# **Chapter 11 (HAC)**

## Google Colab

In [ ]:
## If using Colab

from google.colab import drive
drive.mount('/content/drive')
#drive.mount('/content/drive', force_remount=True)

import os
os.chdir('/content/drive/MyDrive/Workspace/425')          ## replace Workspace/425 with your folder
%cd /content/drive/MyDrive/Workspace/425

## (1) Toy Data

In [ ]:
import pandas as pd

ToyDF = pd.read_csv('./data/HAC_toydata.csv')
ToyDF

In [ ]:
### Set point as row index (excluded from distance calculation)

ToyDF = ToyDF.set_index('point')
ToyDF

## HAC Steps
1. Get lingkage matrix (i.e. HAC result) -->  **[Manual: scipy.cluster.hierarchy.linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html#scipy.cluster.hierarchy.linkage)**
2. Plot dendrogram from linkage matrix --> **[Manual: scipy.cluster.hierarchy.dendrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html#scipy.cluster.hierarchy.dendrogram)**
3. Get cophenetic correlation from linkage matrix and original distance matrix --> **[Manual: scipy.cluster.hierarchy.cophenet](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.cophenet.html#scipy.cluster.hierarchy.cophenet)**

In [ ]:
from scipy.cluster          import hierarchy
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
import seaborn as sns

def runHAC(df):
    linkage_params = ['single', 'complete', 'average', 'centroid', 'median']

    for link in linkage_params:
        ## (1) Calculate linkage matrix Z
        Z = hierarchy.linkage(df, link)
        #print("----- Linkage = ", link, " -----")
        #print(Z)


        ## (2) Draw dendrogram from Z
        plt.figure( figsize = (5, 3) )
        plt.title(link + ' linkage')
        dendro = hierarchy.dendrogram(Z, labels = df.index,
                                      leaf_rotation = 90, leaf_font_size = 9)


        ## (3) Calculate copenetic correlation between Z and real distances
        coph_corr, coph_dist = hierarchy.cophenet(Z, pdist(df))
        print("Cophenetic correlation, linkage = ", link, " >> %.4f \n" % coph_corr)

In [ ]:
runHAC(ToyDF)

### Alternative Visualization (dendrogram + heatmap)

In [ ]:
def seabornHAC(df):
    linkage_params = ['single', 'complete', 'average', 'centroid', 'median']

    for link in linkage_params:
        dendroheat = sns.clustermap(df, method = link, col_cluster = False, figsize = (6,6))
        dendroheat.ax_heatmap.set_title(link + ' linkage')

In [ ]:
seabornHAC(ToyDF)

## (2) Dentition Data

In [ ]:
import pandas as pd

DentitionDF = pd.read_csv('./data/dentition.csv')
print(DentitionDF, "\n")
print(DentitionDF.info())

In [ ]:
### Set name as row index (excluded from distance calculation)

DentitionDF = DentitionDF.set_index('name')
print(DentitionDF, "\n")
print(DentitionDF.info())

In [ ]:
runHAC(DentitionDF)

In [ ]:
seabornHAC(DentitionDF)

## DBSCAN Steps
1. Fit DBSCAN --> **[Manual: sklearn.cluster.DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)**
2. Visualize clusters by scatter plot

Optional preprocessing
- Standardization, if attribute ranges are much different
- PCA for easy visualization --> **[Manual: sklearn.decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)**

In [ ]:
import pandas as pd

IrisDF = pd.read_excel('./data/IrisFromWeka.xlsx')

IrisDF2 = IrisDF.drop('class', axis=1)     ## don't use class for clustering
IrisDF2.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

cols    = IrisDF2.columns
scaler  = StandardScaler()
IrisDF2 = scaler.fit_transform(IrisDF2)
IrisDF2 = pd.DataFrame(IrisDF2, columns = cols)
IrisDF2.head()

In [ ]:
from sklearn.decomposition import PCA

pca       = PCA(n_components = 2)
newvalues = pca.fit_transform(IrisDF2)
df = pd.DataFrame(data = newvalues, columns = ['PC1', 'PC2'])
df.head()

In [ ]:
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

def runDBSCAN(df, epsIn, minPts):
  out     = DBSCAN(eps = epsIn, min_samples = minPts).fit(df)
  labels  = out.labels_         ## cluster assignment for each data point
  clusIDs = set(labels)         ## cluster IDs

  print("Clusters =", clusIDs)
  print("Labels   =", labels, "\n")

  clusters = {}
  plt.figure(figsize = (4, 4))

  for id in clusIDs:
      clusters[id] = df[labels == id]
      print("Cluster ", id, " = ", len(clusters[id]))
      plt.scatter(clusters[id]['PC1'], clusters[id]['PC2'])

  plt.title("DBSCAN (eps = " + str(epsIn) + ", minPts = " + str(minPts) + ")")
  plt.xlabel("PC1")
  plt.ylabel("PC2")
  plt.legend(clusters.keys())
  plt.show()

In [ ]:
runDBSCAN(df, 0.4, 5)

In [ ]:
runDBSCAN(df, 0.6, 10)